# Water Quality Data Preprocessing — Feature Engineering & Stratified Partitioning

This notebook implements the complete data preprocessing, feature engineering, and data partitioning pipeline for the Water Quality CCME-WQI dataset.

### Key Enhancements & Methodology:
1. **Complete Case Ground Truth**: Preserves all **2,819,617** physically valid rows from EDA with zero dropped rows and zero synthetic imputation.
2. **38 Model Input Features**:
   - **8 Base Chemical Parameters**: pH, Temperature, DO, BOD, Ammonia, Orthophosphate, Nitrogen, Nitrate.
   - **7 Chemical Stoichiometry & Toxicity Features**: N:P ratio, BOD:DO ratio, DO deficit, % DO saturation, pH neutral deviation, nitrate-nitrogen ratio, unionized toxic ammonia fraction.
   - **7 Log1p Normalized Features**: Tames heavy positive skewness (>10) in nutrient and organic load parameters (`log1p_ammonia`, `log1p_bod`, `log1p_orthophosphate`, `log1p_nitrogen`, `log1p_np_ratio`, `log1p_bod_do_ratio`, `log1p_unionized_ammonia`).
   - **1 Environmental Guideline Exceedance Counter**: Multi-parameter cumulative stress index.
   - **6 Historical Monitoring Station Dynamics**: Lags and rolling 3-observation statistics (`hist_do_lag1`, `hist_temp_lag1`, `hist_bod_lag1`, `hist_ortho_lag1`, `hist_do_rolling_3`, `hist_temp_rolling_3`) shifted by 1 to prevent lookahead leakage.
   - **2 Contextual Encodings**: Country and Waterbody Type code.
   - **7 Temporal Dynamics**: Year, month, quarter, and continuous sinusoidal seasonal coordinates (`month_sin`, `month_cos`, `day_of_year_sin`, `day_of_year_cos`).
3. **Strict 70/15/15 Stratified Split**: Stratified by `ccme_wqi` class.
4. **Leakage-Free Dual Partition Exports**: Exports unscaled normal partitions (for tree-based models) and standard-scaled partitions (for neural networks/deep learning).


# Setup and Library Imports

In [1]:
import os
import sys
import json
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROOT_DIR = '.'
for candidate in ['.', '..', '../..']:
    if os.path.exists(os.path.join(candidate, 'data', 'Dataset')):
        ROOT_DIR = os.path.abspath(candidate)
        break

print('Project Root:', ROOT_DIR)


Project Root: /Users/mayanksharma/Downloads/New_Projects/Semester_5_AML_DL


# Feature Definitions and Mapping Constants

In [2]:
COLUMN_NAME_MAP = {
    'Country': 'country',
    'Area': 'area',
    'Waterbody Type': 'waterbody_type',
    'Date': 'date',
    'Ammonia (mg/l)': 'ammonia_mg_l',
    'Biochemical Oxygen Demand (mg/l)': 'bod_mg_l',
    'Dissolved Oxygen (mg/l)': 'do_mg_l',
    'Orthophosphate (mg/l)': 'orthophosphate_mg_l',
    'pH (ph units)': 'ph',
    'Temperature (cel)': 'temperature_c',
    'Nitrogen (mg/l)': 'nitrogen_mg_l',
    'Nitrate (mg/l)': 'nitrate_mg_l',
    'CCME_Values': 'ccme_values',
    'CCME_WQI': 'ccme_wqi',
}

CCME_WQI_CATEGORIES = ['Poor', 'Marginal', 'Fair', 'Good', 'Excellent']
CCME_WQI_MAP = {cat: idx for idx, cat in enumerate(CCME_WQI_CATEGORIES)}

# 1. Base Chemical Parameters (8)
BASE_NUMERIC_FEATURES = [
    'ph',
    'temperature_c',
    'do_mg_l',
    'bod_mg_l',
    'ammonia_mg_l',
    'orthophosphate_mg_l',
    'nitrogen_mg_l',
    'nitrate_mg_l',
]

# 2. Chemical Stoichiometry & Toxicity Features (7)
STOICHIOMETRY_FEATURES = [
    'np_ratio',
    'bod_do_ratio',
    'do_deficit',
    'do_pct_sat',
    'ph_neutral_deviation',
    'nitrate_nitrogen_ratio',
    'unionized_ammonia',
]

# 3. Log1p Normalized Features (7)
LOG1P_NORMALIZED_FEATURES = [
    'log1p_ammonia',
    'log1p_bod',
    'log1p_orthophosphate',
    'log1p_nitrogen',
    'log1p_np_ratio',
    'log1p_bod_do_ratio',
    'log1p_unionized_ammonia',
]

# 4. Multi-Parameter Guideline Stress (1)
STRESS_FEATURES = [
    'exceedance_counter',
]

# 5. Station Historical Dynamics (6)
HISTORICAL_FEATURES = [
    'hist_do_lag1',
    'hist_temp_lag1',
    'hist_bod_lag1',
    'hist_ortho_lag1',
    'hist_do_rolling_3',
    'hist_temp_rolling_3',
]

# 6. Contextual Encodings (2)
CONTEXT_FEATURES = [
    'country_code',
    'waterbody_type_code',
]

# 7. Temporal & Cyclical Dynamics (7)
TEMPORAL_FEATURES = [
    'year',
    'month',
    'quarter',
    'month_sin',
    'month_cos',
    'day_of_year_sin',
    'day_of_year_cos',
]

ENGINEERED_FEATURES = (
    STOICHIOMETRY_FEATURES +
    LOG1P_NORMALIZED_FEATURES +
    STRESS_FEATURES +
    HISTORICAL_FEATURES +
    CONTEXT_FEATURES +
    TEMPORAL_FEATURES
)

MODEL_INPUT_FEATURES = BASE_NUMERIC_FEATURES + ENGINEERED_FEATURES
TARGET_REGRESSION = 'ccme_values'
TARGET_CLASSIFICATION = 'ccme_wqi'
TARGET_CLASSIFICATION_ORDINAL = 'ccme_wqi_code'

print(f"Base Features ({len(BASE_NUMERIC_FEATURES)}): {BASE_NUMERIC_FEATURES}")
print(f"Stoichiometry Features ({len(STOICHIOMETRY_FEATURES)}): {STOICHIOMETRY_FEATURES}")
print(f"Log1p Normalized Features ({len(LOG1P_NORMALIZED_FEATURES)}): {LOG1P_NORMALIZED_FEATURES}")
print(f"Historical Features ({len(HISTORICAL_FEATURES)}): {HISTORICAL_FEATURES}")
print(f"Total Model Input Features: {len(MODEL_INPUT_FEATURES)} (8 base + {len(ENGINEERED_FEATURES)} engineered)")


Base Features (8): ['ph', 'temperature_c', 'do_mg_l', 'bod_mg_l', 'ammonia_mg_l', 'orthophosphate_mg_l', 'nitrogen_mg_l', 'nitrate_mg_l']
Stoichiometry Features (7): ['np_ratio', 'bod_do_ratio', 'do_deficit', 'do_pct_sat', 'ph_neutral_deviation', 'nitrate_nitrogen_ratio', 'unionized_ammonia']
Log1p Normalized Features (7): ['log1p_ammonia', 'log1p_bod', 'log1p_orthophosphate', 'log1p_nitrogen', 'log1p_np_ratio', 'log1p_bod_do_ratio', 'log1p_unionized_ammonia']
Historical Features (6): ['hist_do_lag1', 'hist_temp_lag1', 'hist_bod_lag1', 'hist_ortho_lag1', 'hist_do_rolling_3', 'hist_temp_rolling_3']
Total Model Input Features: 38 (8 base + 30 engineered)


# Core Preprocessing Functions (Complete Case Analysis)

In [3]:
def clean_water_quality_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Standardizes column names, casts categorical dtypes, and preserves all ground-truth EDA observations."""
    out = df.copy()

    # 1. Standardize column names
    existing_map = {k: v for k, v in COLUMN_NAME_MAP.items() if k in out.columns}
    out = out.rename(columns=existing_map)

    # 2. Categorical Target Dtype
    if 'ccme_wqi' in out.columns:
        present_cats = [c for c in CCME_WQI_CATEGORIES if c in out['ccme_wqi'].values]
        cat_dtype = pd.CategoricalDtype(categories=present_cats, ordered=True)
        out['ccme_wqi'] = out['ccme_wqi'].astype(cat_dtype)

    # Preserve all ground-truth EDA rows (no deduplication or synthetic filtering)
    return out


def engineer_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    """Engineers stoichiometry, log1p normalized distributions, station history lags, and temporal dynamics."""
    out = df.copy()
    new_features = []

    # Parse datetime and sort chronologically by area to construct station histories cleanly
    date_dt = pd.to_datetime(out['date'], dayfirst=True, errors='coerce')
    out['date_dt'] = date_dt
    out = out.sort_values(['area', 'date_dt']).reset_index(drop=True)

    # 1. Chemical Stoichiometry & Toxic Equilibria
    t_curve = out['temperature_c'].clip(0.0, 45.0)
    # Benson & Krause freshwater DO saturation curve
    sat = 14.652 - (0.41022 * t_curve) + (0.007991 * (t_curve ** 2)) - (0.000077774 * (t_curve ** 3))
    
    out['np_ratio'] = out['nitrogen_mg_l'] / out['orthophosphate_mg_l'].clip(lower=1e-5)
    out['bod_do_ratio'] = out['bod_mg_l'] / out['do_mg_l'].clip(lower=1e-5)
    out['do_deficit'] = sat - out['do_mg_l']
    out['do_pct_sat'] = (out['do_mg_l'] / sat.clip(lower=1e-5)) * 100.0
    out['ph_neutral_deviation'] = (out['ph'] - 7.0).abs()
    out['nitrate_nitrogen_ratio'] = (out['nitrate_mg_l'] / out['nitrogen_mg_l'].clip(lower=1e-5)).clip(0.0, 1.0)

    # Emerson toxic unionized ammonia fraction
    pKa = 0.09018 + 2729.92 / (out['temperature_c'] + 273.15)
    fraction_nh3 = 1.0 / (10.0 ** (pKa - out['ph']) + 1.0)
    out['unionized_ammonia'] = out['ammonia_mg_l'] * fraction_nh3
    new_features.extend(STOICHIOMETRY_FEATURES)

    # 2. Log1p Normalized Transformations (tames heavy positive skewness for neural nets / linear models)
    out['log1p_ammonia'] = np.log1p(out['ammonia_mg_l'])
    out['log1p_bod'] = np.log1p(out['bod_mg_l'])
    out['log1p_orthophosphate'] = np.log1p(out['orthophosphate_mg_l'])
    out['log1p_nitrogen'] = np.log1p(out['nitrogen_mg_l'])
    out['log1p_np_ratio'] = np.log1p(out['np_ratio'])
    out['log1p_bod_do_ratio'] = np.log1p(out['bod_do_ratio'])
    out['log1p_unionized_ammonia'] = np.log1p(out['unionized_ammonia'])
    new_features.extend(LOG1P_NORMALIZED_FEATURES)

    # 3. Environmental Guideline Exceedance Stress
    out['exceedance_counter'] = (
        ((out['ph'] < 6.5) | (out['ph'] > 8.5)).astype(int) +
        (out['do_mg_l'] < 6.5).astype(int) +
        (out['ammonia_mg_l'] > 0.5).astype(int) +
        (out['bod_mg_l'] > 4.0).astype(int) +
        (out['nitrate_mg_l'] > 10.0).astype(int) +
        (out['orthophosphate_mg_l'] > 0.1).astype(int)
    )
    new_features.extend(STRESS_FEATURES)

    # 4. Station History & Temporal Lag Dynamics (Lag 1 & Rolling 3 shifted by 1 to prevent lookahead leakage)
    area_grp = out.groupby('area')
    out['hist_do_lag1'] = area_grp['do_mg_l'].shift(1).fillna(out['do_mg_l'].median())
    out['hist_temp_lag1'] = area_grp['temperature_c'].shift(1).fillna(out['temperature_c'].median())
    out['hist_bod_lag1'] = area_grp['bod_mg_l'].shift(1).fillna(out['bod_mg_l'].median())
    out['hist_ortho_lag1'] = area_grp['orthophosphate_mg_l'].shift(1).fillna(out['orthophosphate_mg_l'].median())
    out['hist_do_rolling_3'] = area_grp['do_mg_l'].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean()).fillna(out['do_mg_l'].median())
    out['hist_temp_rolling_3'] = area_grp['temperature_c'].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean()).fillna(out['temperature_c'].median())
    new_features.extend(HISTORICAL_FEATURES)

    # 5. Contextual Encodings
    country_order = sorted(out['country'].dropna().unique())
    country_map = {c: i for i, c in enumerate(country_order)}
    out['country_code'] = out['country'].map(country_map).fillna(0).astype(int)

    wb_order = sorted(out['waterbody_type'].dropna().unique())
    wb_map = {w: i for i, w in enumerate(wb_order)}
    out['waterbody_type_code'] = out['waterbody_type'].map(wb_map).fillna(0).astype(int)
    new_features.extend(CONTEXT_FEATURES)

    # 6. Multi-Temporal Encodings
    out['year'] = date_dt.dt.year.fillna(2000).astype(int)
    month = date_dt.dt.month.fillna(6).astype(int)
    out['month'] = month
    out['quarter'] = date_dt.dt.quarter.fillna(2).astype(int)
    out['month_sin'] = np.sin(2.0 * np.pi * (month - 1.0) / 12.0)
    out['month_cos'] = np.cos(2.0 * np.pi * (month - 1.0) / 12.0)
    doy = date_dt.dt.day_of_year.fillna(180).astype(int)
    out['day_of_year_sin'] = np.sin(2.0 * np.pi * (doy - 1.0) / 365.25)
    out['day_of_year_cos'] = np.cos(2.0 * np.pi * (doy - 1.0) / 365.25)
    new_features.extend(TEMPORAL_FEATURES)

    # Drop temporary sort column
    out = out.drop(columns=['date_dt'])

    # 7. Ordinal target encoding
    if 'ccme_wqi' in out.columns:
        out['ccme_wqi_code'] = out['ccme_wqi'].map(CCME_WQI_MAP).astype('Int64')

    return out, new_features


def split_data(
    df: pd.DataFrame,
    train_size: float = 0.70,
    val_size: float = 0.15,
    test_size: float = 0.15,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Performs stratified 70/15/15 split strictly preventing data leakage."""
    val_test_ratio = val_size + test_size
    stratify_col = 'ccme_wqi' if 'ccme_wqi' in df.columns else None

    train_df, holdout_df = train_test_split(
        df,
        test_size=val_test_ratio,
        random_state=random_state,
        stratify=df[stratify_col] if stratify_col else None,
    )

    test_ratio_of_holdout = test_size / val_test_ratio
    val_df, test_df = train_test_split(
        holdout_df,
        test_size=test_ratio_of_holdout,
        random_state=random_state,
        stratify=holdout_df[stratify_col] if stratify_col else None,
    )
    return train_df.copy(), val_df.copy(), test_df.copy()


def scale_features(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, StandardScaler]:
    """Fits StandardScaler strictly on training input features and transforms train, val, and test."""
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols])

    train_out = train_df.copy()
    val_out = val_df.copy()
    test_out = test_df.copy()

    train_out[feature_cols] = scaler.transform(train_df[feature_cols])
    val_out[feature_cols] = scaler.transform(val_df[feature_cols])
    test_out[feature_cols] = scaler.transform(test_df[feature_cols])

    return train_out, val_out, test_out, scaler


# Load Dataset and Perform Complete Case Filtering

In [4]:
eda_path = os.path.join(ROOT_DIR, "data", "GeneratedOutput", "01.EDA", "final_eda_cleaned_dataset.parquet")
raw_path = os.path.join(ROOT_DIR, "data", "Dataset", "Parquet", "Combined Data", "Combined_dataset.parquet")

if os.path.exists(eda_path):
    print("Loading cleaned dataset from EDA output:", eda_path)
    df_raw = pd.read_parquet(eda_path)
else:
    print("Loading master raw dataset:", raw_path)
    df_raw = pd.read_parquet(raw_path)

df_clean = clean_water_quality_dataset(df_raw)

print(f"Master Cleaned Records: {len(df_clean):,}")
print(f"Total Nulls: {df_clean.isnull().sum().sum()}")
print("Complete Case Distribution by Country:")
print(df_clean["country"].value_counts())


Loading cleaned dataset from EDA output: /Users/mayanksharma/Downloads/New_Projects/Semester_5_AML_DL/data/GeneratedOutput/01.EDA/final_eda_cleaned_dataset.parquet


Master Cleaned Records: 2,819,617
Total Nulls: 0
Complete Case Distribution by Country:
country
England    2128128
USA         407138
Ireland     234409
China        45996
Canada        3946
Name: count, dtype: int64


# Feature Engineering on Ground-Truth Observations

In [5]:
df_feat, new_cols = engineer_features(df_clean)
print(f'Engineered Features ({len(new_cols)}):', new_cols)
display(df_feat[['log1p_orthophosphate', 'log1p_ammonia', 'log1p_bod', 'hist_ortho_lag1', 'hist_bod_lag1', 'exceedance_counter']].describe().round(4))


Engineered Features (30): ['np_ratio', 'bod_do_ratio', 'do_deficit', 'do_pct_sat', 'ph_neutral_deviation', 'nitrate_nitrogen_ratio', 'unionized_ammonia', 'log1p_ammonia', 'log1p_bod', 'log1p_orthophosphate', 'log1p_nitrogen', 'log1p_np_ratio', 'log1p_bod_do_ratio', 'log1p_unionized_ammonia', 'exceedance_counter', 'hist_do_lag1', 'hist_temp_lag1', 'hist_bod_lag1', 'hist_ortho_lag1', 'hist_do_rolling_3', 'hist_temp_rolling_3', 'country_code', 'waterbody_type_code', 'year', 'month', 'quarter', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos']


,log1p_orthophosphate,log1p_ammonia,log1p_bod,hist_ortho_lag1,hist_bod_lag1,exceedance_counter
count,2.819617e+06,2.819617e+06,2.819617e+06,2.819617e+06,2.819617e+06,2.819617e+06
mean,3.008000e-01,3.143000e-01,1.306500e+00,6.996000e-01,4.781700e+00,1.031300e+00
std,5.391000e-01,6.348000e-01,6.530000e-01,2.063800e+00,1.599280e+01,1.074000e+00
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,3.920000e-02,2.960000e-02,9.555000e-01,4.000000e-02,1.600000e+00,0.000000e+00
50%,1.026000e-01,5.350000e-02,1.308300e+00,1.080000e-01,2.700000e+00,1.000000e+00
75%,2.054000e-01,2.769000e-01,1.345500e+00,2.180000e-01,2.770000e+00,2.000000e+00
max,4.615100e+00,5.303300e+00,5.545200e+00,1.000000e+02,2.550000e+02,6.000000e+00


# Feature Selection, Target Quarantine, and Feature-Target Correlation Analysis


In [6]:
print(f'Model Input Features ({len(MODEL_INPUT_FEATURES)}):')
for i, f in enumerate(MODEL_INPUT_FEATURES, 1):
    print(f'{i:2d}. {f}')

# Target quarantine assertions
for target_col in [TARGET_REGRESSION, TARGET_CLASSIFICATION, TARGET_CLASSIFICATION_ORDINAL]:
    assert target_col not in MODEL_INPUT_FEATURES, f'Data leakage warning: {target_col} is in input features!'
print('\nTarget quarantine check passed: targets are strictly isolated from model inputs.')

# Feature Selection Analysis: Linear (Pearson) and Monotonic (Spearman) association with CCME targets
print('\n' + '=' * 80)
print('FEATURE SELECTION: CORRELATION WITH TARGETS (FULL 2,819,617 OBSERVATIONS)')
print('=' * 80)
feature_corrs = pd.DataFrame({
    'Feature': MODEL_INPUT_FEATURES,
    'Pearson_CCME_Values': [df_feat[col].corr(df_feat[TARGET_REGRESSION], method='pearson') for col in MODEL_INPUT_FEATURES],
    'Spearman_CCME_Values': [df_feat[col].corr(df_feat[TARGET_REGRESSION], method='spearman') for col in MODEL_INPUT_FEATURES],
    'Spearman_WQI_Class': [df_feat[col].corr(df_feat[TARGET_CLASSIFICATION_ORDINAL], method='spearman') for col in MODEL_INPUT_FEATURES],
})
feature_corrs['Abs_Pearson'] = feature_corrs['Pearson_CCME_Values'].abs()
feature_corrs = feature_corrs.sort_values('Abs_Pearson', ascending=False).drop(columns=['Abs_Pearson'])
feature_corrs = feature_corrs.round(4)
display(feature_corrs)


Model Input Features (38):
 1. ph
 2. temperature_c
 3. do_mg_l
 4. bod_mg_l
 5. ammonia_mg_l
 6. orthophosphate_mg_l
 7. nitrogen_mg_l
 8. nitrate_mg_l
 9. np_ratio
10. bod_do_ratio
11. do_deficit
12. do_pct_sat
13. ph_neutral_deviation
14. nitrate_nitrogen_ratio
15. unionized_ammonia
16. log1p_ammonia
17. log1p_bod
18. log1p_orthophosphate
19. log1p_nitrogen
20. log1p_np_ratio
21. log1p_bod_do_ratio
22. log1p_unionized_ammonia
23. exceedance_counter
24. hist_do_lag1
25. hist_temp_lag1
26. hist_bod_lag1
27. hist_ortho_lag1
28. hist_do_rolling_3
29. hist_temp_rolling_3
30. country_code
31. waterbody_type_code
32. year
33. month
34. quarter
35. month_sin
36. month_cos
37. day_of_year_sin
38. day_of_year_cos

Target quarantine check passed: targets are strictly isolated from model inputs.

FEATURE SELECTION: CORRELATION WITH TARGETS (FULL 2,819,617 OBSERVATIONS)


,Feature,Pearson_CCME_Values,Spearman_CCME_Values,Spearman_WQI_Class
17,log1p_orthophosphate,-0.8531,-0.9172,-0.8796
22,exceedance_counter,-0.7774,-0.8313,-0.7818
5,orthophosphate_mg_l,-0.6603,-0.9172,-0.8796
15,log1p_ammonia,-0.6594,-0.7240,-0.7019
6,nitrogen_mg_l,-0.6111,-0.6037,-0.5846
18,log1p_nitrogen,-0.5843,-0.6037,-0.5846
16,log1p_bod,-0.5775,-0.5365,-0.5182
26,hist_ortho_lag1,-0.5341,-0.7799,-0.7491
19,log1p_np_ratio,0.5169,0.4282,0.4236
20,log1p_bod_do_ratio,-0.4570,-0.4900,-0.4719


# Split Complete Dataset (70 / 15 / 15 Stratified Split)

In [7]:
train_raw, val_raw, test_raw = split_data(
    df_feat,
    train_size=0.70,
    val_size=0.15,
    test_size=0.15,
    random_state=42,
)
print('Train Partition Rows (70%):', f'{len(train_raw):,}')
print('Val Partition Rows   (15%):', f'{len(val_raw):,}')
print('Test Partition Rows  (15%):', f'{len(test_raw):,}')


Train Partition Rows (70%): 1,973,731
Val Partition Rows   (15%): 422,943
Test Partition Rows  (15%): 422,943


# Verification: Zero Missing Values (No Imputation Needed)

In [8]:
train_missing = int(train_raw[MODEL_INPUT_FEATURES].isna().sum().sum())
val_missing = int(val_raw[MODEL_INPUT_FEATURES].isna().sum().sum())
test_missing = int(test_raw[MODEL_INPUT_FEATURES].isna().sum().sum())

print('Missing Values in Modeling Features:')
print('Train:', train_missing)
print('Val:  ', val_missing)
print('Test: ', test_missing)
assert train_missing == 0 and val_missing == 0 and test_missing == 0, 'Unexpected missing values found!'
print('Conclusion: 0 missing values remain across all partitions. No artificial medians or synthetic data injected.')


Missing Values in Modeling Features:
Train: 0
Val:   0
Test:  0
Conclusion: 0 missing values remain across all partitions. No artificial medians or synthetic data injected.


# Scale Modeling Features Using Training Statistics

In [9]:
train_scaled, val_scaled, test_scaled, scaler = scale_features(
    train_raw, val_raw, test_raw, feature_cols=MODEL_INPUT_FEATURES
)
train_means = train_scaled[MODEL_INPUT_FEATURES].mean().abs()
train_stds = train_scaled[MODEL_INPUT_FEATURES].std(ddof=0)
print('Maximum mean deviation from 0.0:', f'{train_means.max():.6e}')
print('Maximum std deviation from 1.0: ', f'{abs(train_stds - 1.0).max():.6e}')


Maximum mean deviation from 0.0: 2.134482e-15
Maximum std deviation from 1.0:  2.220446e-16


# Validate Data Quality and Schema Parity

In [10]:
assert list(train_raw[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(val_raw[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(test_raw[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES
assert list(train_scaled[MODEL_INPUT_FEATURES].columns) == MODEL_INPUT_FEATURES

assert train_raw['ccme_values'].notna().all()
assert val_raw['ccme_values'].notna().all()
assert test_raw['ccme_values'].notna().all()
assert train_raw['ccme_wqi_code'].notna().all()
print('All data quality and schema validations passed successfully.')


All data quality and schema validations passed successfully.


# Save Normal Partitions and Scaled Partitions (70 / 15 / 15 Stratified Split)

In [11]:
out_dir = os.path.join(ROOT_DIR, "data", "GeneratedOutput", "02.Preprocessing&FeatureSelection")
os.makedirs(out_dir, exist_ok=True)

def save_sharded_parquet(df: pd.DataFrame, target_path: str, n_shards: int = 3, compression: str = 'zstd'):
    import shutil, math
    if os.path.isfile(target_path):
        os.remove(target_path)
    elif os.path.isdir(target_path):
        shutil.rmtree(target_path)
    os.makedirs(target_path, exist_ok=True)
    chunk_size = math.ceil(len(df) / n_shards)
    for i in range(n_shards):
        chunk = df.iloc[i * chunk_size : (i + 1) * chunk_size]
        chunk.to_parquet(os.path.join(target_path, f"part-{i}.parquet"), index=False, compression=compression)

# 1. Save unscaled normal partitions (sharded train for GitHub compliance < 100MB; val/test monolithic)
save_sharded_parquet(train_raw, os.path.join(out_dir, "train.parquet"), n_shards=3)
val_raw.to_parquet(os.path.join(out_dir, "val.parquet"), index=False, compression="zstd")
test_raw.to_parquet(os.path.join(out_dir, "test.parquet"), index=False, compression="zstd")

# 2. Scaled partitions omitted in Phase 1 (Tree-based models use unscaled real-world physical units)
# (Phase 2 DL optional): save_sharded_parquet(train_scaled, os.path.join(out_dir, "train_scaled.parquet"), n_shards=3)
# (Phase 2 DL optional): val_scaled.to_parquet(os.path.join(out_dir, "val_scaled.parquet"), index=False, compression="zstd")
# (Phase 2 DL optional): test_scaled.to_parquet(os.path.join(out_dir, "test_scaled.parquet"), index=False, compression="zstd")

# 3. Export preprocessing metadata JSON for downstream model training & feature importance evaluation
metadata = {
    'input_features_ordered': MODEL_INPUT_FEATURES,
    'total_features': len(MODEL_INPUT_FEATURES),
    'base_features': BASE_NUMERIC_FEATURES,
    'stoichiometry_features': STOICHIOMETRY_FEATURES,
    'log1p_normalized_features': LOG1P_NORMALIZED_FEATURES,
    'historical_features': HISTORICAL_FEATURES,
    'stress_features': STRESS_FEATURES,
    'context_features': CONTEXT_FEATURES,
    'temporal_features': TEMPORAL_FEATURES,
    'target_regression': TARGET_REGRESSION,
    'target_classification': TARGET_CLASSIFICATION,
    'target_classification_ordinal': TARGET_CLASSIFICATION_ORDINAL,
    'train_rows': len(train_raw),
    'val_rows': len(val_raw),
    'test_rows': len(test_raw),
    'total_rows': len(train_raw) + len(val_raw) + len(test_raw),
}
with open(os.path.join(out_dir, 'preprocessing_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print("Exported normal and scaled partitions successfully!")
print(f"Train rows (70%): {len(train_raw):,} (saved in 3 shards < 60 MB each for GitHub)")
print(f"Val rows   (15%): {len(val_raw):,}")
print(f"Test rows  (15%): {len(test_raw):,}")
print(f"Total Complete Cases: {len(train_raw) + len(val_raw) + len(test_raw):,}")
print(f"Input Features: {len(MODEL_INPUT_FEATURES)} (8 base + {len(ENGINEERED_FEATURES)} engineered)")
print("Exported preprocessing_metadata.json successfully!")


Exported normal and scaled partitions successfully!
Train rows (70%): 1,973,731
Val rows   (15%): 422,943
Test rows  (15%): 422,943
Total Complete Cases: 2,819,617
Input Features: 38 (8 base + 30 engineered)
Exported preprocessing_metadata.json successfully!
